# PhysNetJax Training Example

This notebook demonstrates how to run training using `train_model` from `mmml.physnetjax.physnetjax.training.training`.

## 1. Imports and Setup

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".99"

import jax
import jax.numpy as jnp
import numpy as np

from mmml.physnetjax.physnetjax.models.model import EF
from mmml.physnetjax.physnetjax.training.training import train_model
from mmml.physnetjax.physnetjax.data.data import prepare_datasets

print("JAX devices:", jax.devices())

## 2. Prepare Data

**Option A:** Load from an NPZ file (requires R, Z, F, E, N keys):

In [ ]:
# Option A: Load from NPZ file
# data_path = "path/to/your/dataset.npz"
# key = jax.random.PRNGKey(0)
# train_data, valid_data = prepare_datasets(
#     key,
#     train_size=800,
#     valid_size=100,
#     files=data_path,
#     natoms=60,
#     verbose=True,
# )

**Option B:** Create minimal synthetic data for a quick test run:

In [ ]:
# Option B: Synthetic data for quick testing
def make_synthetic_data(n_train, n_valid, num_atoms=10):
    """Create minimal train/valid dicts with R, Z, F, E, N."""
    key = jax.random.PRNGKey(42)
    key, k1, k2 = jax.random.split(key, 3)
    
    def _sample(n):
        k = jax.random.split(key, 5)
        R = jax.random.normal(k[0], (n, num_atoms, 3)) * 2.0
        Z = jnp.tile(jnp.array([1, 6, 1, 6, 1, 6, 1, 8, 1, 8]), (n, 1))[:, :num_atoms]  # H, C, O
        Z = jnp.maximum(Z, 1)
        F = jax.random.normal(k[1], (n, num_atoms, 3)) * 0.1
        E = jax.random.normal(k[2], (n, 1)) * 10.0
        N = jnp.full((n,), num_atoms, dtype=jnp.int32)
        return {"R": np.array(R), "Z": np.array(Z), "F": np.array(F), "E": np.array(E), "N": np.array(N)}
    
    train_data = _sample(n_train)
    valid_data = _sample(n_valid)
    return train_data, valid_data

num_atoms = 10
n_train, n_valid = 64, 16
train_data, valid_data = make_synthetic_data(n_train, n_valid, num_atoms=num_atoms)

print("Train R:", train_data["R"].shape)
print("Valid R:", valid_data["R"].shape)

## 3. Create Model and Run Training

In [ ]:
key = jax.random.PRNGKey(0)

model = EF(
    features=16,
    max_degree=2,
    num_iterations=2,
    num_basis_functions=8,
    cutoff=6.0,
    charges=False,
    natoms=num_atoms,
    zbl=False,  # Disable for faster small runs
)

ema_params, best_loss = train_model(
    key=key,
    model=model,
    train_data=train_data,
    valid_data=valid_data,
    num_epochs=5,
    learning_rate=1e-3,
    batch_size=8,
    num_atoms=num_atoms,
    batch_method="default",
    data_keys=("R", "Z", "F", "E", "N"),
    name="physnetjax-notebook-example",
    best=True,
    log_tb=False,  # Set True if tensorflow is installed
    print_freq=1,
)

print(f"\nTraining complete. Best loss: {best_loss:.6f}")

## 4. Using Advanced Batching (Variable-Size Molecules)

For datasets with variable molecule sizes, use `batch_method="advanced"` with `batch_args_dict`:

In [ ]:
# Advanced batching example (uncomment to use)
# batch_args = {
#     "batch_shape": (batch_size,),  # e.g. (32,)
#     "batch_nbl_len": max_neighbor_pairs,  # max pairs across batch
#     "nb_len": max_neighbor_pairs,
# }
# ema_params, best_loss = train_model(
#     ...
#     batch_method="advanced",
#     batch_args_dict=batch_args,
# )